# Pet Breed Classification: Custom CNN vs Pre-trained CNN

**Dataset:** Oxford-IIIT Pet Dataset  
**Task:** Multi-class image classification  
**Main aim:** Compare a custom CNN trained from scratch with a pre-trained CNN using transfer learning.

This notebook includes:
1. Dataset loading and exploration  
2. Preprocessing and augmentation  
3. Custom CNN baseline  
4. Pre-trained CNN using MobileNetV2  
5. Evaluation using accuracy, precision, recall, F1-score, classification report, and confusion matrix  
6. Final comparison table  
7. Prediction function for new images  


In [ ]:
# 1. Install / import libraries


import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)


In [4]:

# 2. Load Oxford-IIIT Pet Dataset

# as_supervised=True gives data as (image, label)
dataset, info = tfds.load(
    "oxford_iiit_pet",
    as_supervised=True,
    with_info=True
)

train_full = dataset["train"]
test_raw = dataset["test"]

class_names = info.features["label"].names
NUM_CLASSES = info.features["label"].num_classes

print("Dataset name:", info.name)
print("Number of classes:", NUM_CLASSES)
print("Training examples:", info.splits["train"].num_examples)
print("Test examples:", info.splits["test"].num_examples)
print("First 10 class names:", class_names[:10])

NameError: name 'tfds' is not defined

## Dataset exploration

This section checks the number of images, classes, image examples, and class distribution. This is important because class imbalance or visually similar classes can affect model performance.

In [ ]:

# 3. Visualize sample images


plt.figure(figsize=(12, 12))

for i, (image, label) in enumerate(train_full.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[int(label)])
    plt.axis("off")

plt.suptitle("Sample Images from Oxford-IIIT Pet Dataset", fontsize=16)
plt.show()

In [ ]:

# 4. Check image shapes

for image, label in train_full.take(5):
    print("Image shape:", image.shape, "| Label:", int(label), "| Class:", class_names[int(label)])

In [ ]:
# 5. Class distribution


def get_label_counts(ds, class_names):
    labels = []
    for _, label in tfds.as_numpy(ds):
        labels.append(int(label))
    counts = pd.Series(labels).value_counts().sort_index()
    df = pd.DataFrame({
        "class_id": counts.index,
        "class_name": [class_names[i] for i in counts.index],
        "count": counts.values
    })
    return df

train_counts = get_label_counts(train_full, class_names)
test_counts = get_label_counts(test_raw, class_names)

display(train_counts.head())

plt.figure(figsize=(16, 6))
plt.bar(train_counts["class_name"], train_counts["count"])
plt.xticks(rotation=90)
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.title("Training Set Class Distribution")
plt.tight_layout()
plt.show()

## Preprocessing

Both models use the same image size and the same train/validation/test split.  
This makes the comparison fair.

- Images are resized to 224 × 224.
- Training data is shuffled.
- A validation set is created from the training set.
- Datasets are batched and prefetched for faster training.


In [5]:
# 6. Preprocessing settings


IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

TRAIN_SIZE = info.splits["train"].num_examples
VAL_SIZE = int(0.2 * TRAIN_SIZE)

print("Training size before validation split:", TRAIN_SIZE)
print("Validation size:", VAL_SIZE)
print("Final training size:", TRAIN_SIZE - VAL_SIZE)

NameError: name 'tf' is not defined

In [ ]:
# 7. Resize images


def resize_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image, label

# Shuffle once, then split into train and validation sets
train_shuffled = train_full.shuffle(
    buffer_size=TRAIN_SIZE,
    seed=SEED,
    reshuffle_each_iteration=False
)

val_raw = train_shuffled.take(VAL_SIZE)
train_raw = train_shuffled.skip(VAL_SIZE)

train_ds = (
    train_raw
    .map(resize_image, num_parallel_calls=AUTOTUNE)
    .shuffle(1000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_raw
    .map(resize_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_ds = (
    test_raw
    .map(resize_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Datasets prepared successfully.")

In [ ]:
# 8. Data augmentation layer


data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

# Model 1: Custom CNN

The custom CNN is built from scratch. It starts with random weights and learns all features directly from this dataset.

This model is used as the **baseline** for comparison.

In [ ]:
 #9. Build Custom CNN


def build_custom_cnn(num_classes):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = data_augmentation(inputs)
    x = tf.keras.layers.Rescaling(1./255)(x)

    x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="Custom_CNN")
    return model

custom_cnn = build_custom_cnn(NUM_CLASSES)

custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

custom_cnn.summary()

In [ ]:
# 10. Train Custom CNN


callbacks_custom = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

start_time = time.time()

history_custom = custom_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_custom
)

custom_training_time = time.time() - start_time
print(f"Custom CNN training time: {custom_training_time:.2f} seconds")

In [ ]:
# 11. Plot Custom CNN training curves


def plot_history(history, title):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(8, 5))
    plt.plot(hist["accuracy"], label="Training Accuracy")
    plt.plot(hist["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title + " - Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(hist["loss"], label="Training Loss")
    plt.plot(hist["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " - Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(history_custom, "Custom CNN")